# Gradient based dCRAB optimization of a single spin

In this example, we solve the optimization task in the [previous example](02D_GOAToverGRAPE_TLS.ipynb) by using the [dCRAB](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.92.062343) optimization method. Here we implement a gradient based dCRAB algorithm by using the GOAToverGRAPE method.

## 1. Generate a PWC pulse shape

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import DCRABEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.signal.waveform import FlatTopGaussianFilter

Similar to the previous case, lets define the pulse generator, with the envelope being the `DCRABEnvelope`. We use a `FlatTopGaussianFilter` to ensure that the pulse always starts and ends at zero. Finally, to obtain GRAPE gradient we pixelate the pulse using a `PWCGenerator`.

In [ ]:
t_final = 20e-9
tlist = np.linspace(0, t_final, 40)
eps = 5 * np.pi * 1.0  # initial amplitude of the resonator (in MHz)
eps_max = 10 * eps  # maximum amplitude of the resonator (in MHz)

tone = DCRABEnvelope(
    num_components=2,
    max_frequency=5 * 2 * np.pi,
    amplitude=Quantity(eps * 1e6, -eps_max * 1e6, eps_max * 1e6, name="Amplitude"),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final"),
    seed=18537,
)
smooth_tone = FlatTopGaussianFilter(
    envelopes=tone, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

gen = PWCGenerator(envelopes=[smooth_tone], tlist=tlist, max_amplitude=np.sqrt(2) * eps_max * 1e6)

params = gen.get_parameters()

Here the `DCRABEnvelope` is defined with a seed to make sure that the results obtained are reproducible.

In [ ]:
from plotting import plot_signal

ts = np.linspace(0, t_final, 501)
fig, ax = plt.subplots(1, figsize=(5, 3))
plot_signal(smooth_tone, ts, ax, linestyle="-", label="Smooth")
plot_signal(gen, ts, ax, linestyle="--", label="PWC")
ax.legend(loc=1, frameon=True)
plt.show()

## 2. Define Hamiltonian in the rotating frame of drive

As before we define a single spin in the rotating frame of the drive.

In [ ]:
from paraqeet.model.rotating_frame import RotatingFrameDrive
from paraqeet.model.schroedinger_equation import SchroedingerEquation
from paraqeet.model.system import OpenSystem
from paraqeet.quantity import Array


class SpinRWA(OpenSystem):
    """A Single Spin."""

    def __init__(self, drives=None):
        super().__init__(drives)
        self.sigma_p = jnp.array([[0j, 1], [0, 0]])
        self.dim = 2

    def get_hamiltonian_at_timestep(self, timestep: float) -> Array:
        """Just sigma-X."""
        return self._drives[0].get_value_at_timestep(self.sigma_p, timestep)

    def get_hamiltonian_and_gradient(self, times: Array) -> tuple[Array, Array] | tuple[float, Array]:
        """Gradient is just the drive matrix."""
        return self.get_hamiltonian(times), self._drives[0].get_gradient(self.sigma_p, times)

    def get_hamiltonian_gradient_at_timestep(self, time):
        """Computes the gradient"""
        return self._drives[0].get_gradient_at_timestep(self.sigma_p, time)

    def dimension(self) -> int:
        """Returns the dimension"""
        raise self.d

    def get_jump_operators(self) -> list[tuple[Array, Array]]:
        """Returns an empyt list since we study closed system dynamics"""
        return []

    def get_parameters(self):
        """Returns the parameters, which in this case are only the drive parameters"""
        return self._get_drive_parameters()


drive = RotatingFrameDrive(gen)
spin = SpinRWA(drives=[drive])
model = SchroedingerEquation(
    hamiltonian_func=spin.get_hamiltonian, hamiltonian_and_gradient_func=spin.get_hamiltonian_and_gradient
)

And using GRAPE as the method to propagate and compute the gradients. The propagation $dt$ has to be smaller than the $\Delta t$ of the time grid used for discretization. In this case $\Delta t = 0.5$ ns, so we choose the propagation resolution to be > 2e9.

In [ ]:
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.measurement.utils import overlap_state_vector
from paraqeet.propagation.scipy_expm_grape import ScipyExpmGRAPE
from paraqeet.propagation.utils import grape_operator_sandwich_function_closed

init = np.array([[1.0], [0.0]])  # |0>
target = np.array([[0.0], [1.0]])  # |1>

prop = ScipyExpmGRAPE(
    eom_func=model.get_value,
    eom_and_grad_func=model.get_value_and_gradient,
    resolution=3e9,
    initial_state=init,
    target_state=target,
    operator_sandwich_function=grape_operator_sandwich_function_closed,
)

zeroone = StateTransferFidelityGRAPE(
    propagation_func=prop.propagate,
    propagation_and_gradient_func=prop.get_value_and_gradient,
    target_state=target,
    overlap=overlap_state_vector,
)

In [ ]:
from plotting import plot_signal_and_dynamics

ts = np.linspace(0.0, t_final, 101)
plot_signal_and_dynamics(gen, prop, ts, state_labels=[r"$|0\rangle$", r"$|1\rangle$"]);

## 3. Optimisation
Finally, we define the `DCRABOptimizerGradient` that takes the `GOATOverGRAPE` fideltiy to chains together the GRAPE gradients to compute the gradient wrt the dCRAB envelope

In [ ]:
params = tone.get_parameters()
params

Here we add the parameters from the `DCRABEnvelope` to the `optmap` (except the `t_final` to keep the simulation time constant)

In [ ]:
import tempfile

from paraqeet.file_logger import FileLogger
from paraqeet.measurement.goat_over_grape import GOATOverGRAPE
from paraqeet.optimization_map import OptimizationMap
from paraqeet.optimizers.dcrab_optimizer_gradient import DCRABOptimizerGradient

temp_dir = tempfile.TemporaryDirectory()

file_logger = FileLogger(temp_dir.name)

optmap = OptimizationMap()
optmap.add(tone, [params[0]] + params[2:])
optmap.register_params_with_optimizables()

goat = GOATOverGRAPE(zeroone, gen, prop.resolution)
opt_grad = DCRABOptimizerGradient(
    measure_and_gradient_func=goat.get_value_and_gradient,
    optimization_map=optmap,
    super_iteration_every=150,
    max_super_iteration_num=5,
    print_every_iteration_num=10,
    super_iteration_tol=1e-9,
    seed=19573,
)
opt_grad.logger = file_logger

The `optmap` in this case contains the pulse amplitude and the Fourier coefficients for the optimization.

In [ ]:
optmap

In [ ]:
opt_grad.optimize(np.array([0.0, t_final]))

As the parameters are added with random values, the optimization may not succeed sometimes. If it does not reach a low value restart the optimization. 
Here, we have choosen a seed that converges to the target fidelity.

In [ ]:
opt_grad.set_parameters(opt_grad.best_params)
plot_signal_and_dynamics(gen, prop, ts);

In [ ]:
from plotting import plot_infidelity_vs_evaluation_from_logs

plot_infidelity_vs_evaluation_from_logs(log_path=temp_dir.name + "/opt.log", label="GOAT over GRAPE");

In [ ]:
temp_dir.cleanup()